In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Налаштування пом'якшення помилок

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Бета-реліз нової моделі виконання тепер доступний. Модель направленого виконання надає більше гнучкості при налаштуванні робочого процесу пом'якшення помилок. Дивіться посібник [Модель направленого виконання](/guides/directed-execution-model) для отримання додаткової інформації.


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблений із використанням наступних вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Техніки пом'якшення помилок дозволяють користувачам зменшувати помилки
схем шляхом моделювання шуму пристрою під час виконання. Це зазвичай
призводить до квантового overhead попередньої обробки, пов'язаного з
навчанням моделі, та класичного overhead постобробки для пом'якшення
помилок у необроблених результатах за допомогою згенерованої моделі.

Примітив Estimator підтримує кілька технік пом'якшення помилок, включаючи [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) та [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Дивіться [Техніки пом'якшення та придушення помилок](error-mitigation-and-suppression-techniques) для пояснення кожної з них. При використанні примітивів ви можете вмикати або вимикати окремі методи. Дивіться розділ [Користувацькі налаштування помилок](#advanced-error) для деталей.

> **Note:** Sampler не підтримує пом'якшення помилок, але ви можете використовувати пакет [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) для виконання пом'якшення помилок локально.

Estimator також підтримує `resilience_level`. Рівень стійкості визначає,
наскільки стійкою буде система до помилок. Вищі рівні генерують точніші
результати за рахунок більшого часу обробки. Рівні стійкості можна
використовувати для налаштування компромісу вартість/точність при
застосуванні пом'якшення помилок до вашого запиту примітива. Пом'якшення
помилок зменшує помилки (зсув) у результатах шляхом обробки виходів з
колекції або ансамблю пов'язаних схем. Ступінь зменшення помилок залежить
від застосованого методу. Рівень стійкості абстрагує детальний вибір методу
пом'якшення помилок, дозволяючи користувачам міркувати про компроміс
вартість/точність, який підходить для їхнього застосування.

Враховуючи це, кожен рівень відповідає методу або методам зі зростаючим
рівнем квантового вибіркового overhead, що дає вам можливість
експериментувати з різними компромісами час-точність. Наступна таблиця
показує, які рівні та відповідні методи доступні для кожного з примітивів.

> **Info:** Пом'якшення помилок залежить від задачі, тому техніки, які ви можете
> застосувати, варіюються залежно від того, чи ви вибірково визначаєте
> розподіл, чи генеруєте очікувані значення.

<span id="resilience-table"></span>

Estimator підтримує наступні рівні стійкості. Sampler не підтримує рівні стійкості.

| Resilience Level | Definition                                                                                            | Technique                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | No mitigation                                                                                         | None                                                                      |
| 1 [Default]      | Minimal mitigation costs: Mitigate error associated with readout errors                               | Twirled Readout Error eXtinction  (TREX) measurement twirling             |
| 2                | Medium mitigation costs. Typically reduces bias in estimators, but is not guaranteed to be zero-bias. | Level 1 + Zero Noise Extrapolation (ZNE) and gate twirling

> **Info:** Рівні стійкості наразі перебувають у бета-тестуванні, тому вибірковий
> overhead та якість рішення будуть варіюватися від схеми до схеми. Нові
> функції, розширені опції та інструменти управління будуть випускатися на
> постійній основі. Конкретні методи пом'якшення помилок не гарантовано
> застосовуються на кожному рівні стійкості.

## Налаштування Estimator з рівнями стійкості
Ви можете використовувати рівні стійкості для визначення технік пом'якшення помилок, або ви можете встановити користувацькі техніки окремо, як описано у розділі [Користувацькі налаштування помилок.](#advanced-error)

<details>
<summary>Рівень стійкості 0</summary>

Пом'якшення помилок не застосовується до програми користувача.

</details>

<details>
<summary>Рівень стійкості 1</summary>

Рівень 1 застосовує **пом'якшення помилок зчитування** та **twirling вимірювань** шляхом застосування модельно-незалежної техніки, відомої як
Twirled Readout Error eXtinction (TREX). Вона зменшує помилку
вимірювання шляхом діагоналізації каналу шуму, пов'язаного з
вимірюванням, за допомогою випадкового перемикання кубітів через вентилі X
безпосередньо перед вимірюванням. Масштабуючий коефіцієнт з діагонального
каналу шуму вивчається шляхом бенчмаркінгу випадкових схем,
ініціалізованих у нульовому стані. Це дозволяє сервісу видаляти зсув з
очікуваних значень, що виникає через шум зчитування. Цей підхід
детальніше описаний у [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Рівень стійкості 2</summary>

Рівень 2 застосовує **техніки пом'якшення помилок, включені у рівень 1**, а також застосовує **gate twirling** та використовує **метод екстраполяції нульового шуму (ZNE)**. ZNE обчислює
очікуване значення спостережуваної величини для різних факторів шуму
(етап підсилення), а потім використовує виміряні очікувані значення для
виведення ідеального очікуваного значення на межі нульового шуму (етап
екстраполяції). Цей підхід зазвичай зменшує помилки в очікуваних
значеннях, але не гарантує отримання незміщеного результату.

![This image shows a graph.  The x-axis is labeled Noise amplification factor.  The y-axis is labeled Expectation value.  An upward sloping line is labeled Mitigated value.  Points near the line are noise-amplified values.  There is a horizontal line just above the X-axis labeled Exact value.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Illustration of the ZNE method")

Overhead цього методу масштабується з кількістю факторів шуму.
Налаштування за замовчуванням вибірково визначають очікуване значення
при трьох факторах шуму, що призводить до приблизно 3-кратного overhead
при використанні цього рівня стійкості.

У рівні 2 метод TREX випадковим чином перемикає кубіти через вентилі X безпосередньо перед вимірюванням
та перемикає відповідний виміряний біт, якщо вентиль X було застосовано. Цей підхід детальніше описаний у [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Приклад
Інтерфейс `EstimatorV2` дозволяє користувачам безперешкодно працювати з
різними методами пом'якшення помилок для зменшення помилок в очікуваних
значеннях спостережуваних величин. Наступний код використовує Zero Noise Extrapolation та пом'якшення помилок зчитування шляхом простого
встановлення `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Користувацькі налаштування помилок
Ви можете вмикати та вимикати окремі методи пом'якшення та придушення помилок, включаючи dynamical decoupling, gate та measurement twirling, пом'якшення помилок вимірювання, PEC та ZNE. Дивіться [Техніки пом'якшення та придушення помилок](error-mitigation-and-suppression-techniques) для пояснення кожного з них.

> **Note:** - Не всі опції доступні для обох примітивів. Дивіться [таблицю доступних опцій](runtime-options-overview#options-table) для списку доступних опцій.
> - Не всі методи працюють разом на всіх типах схем. Дивіться [таблицю сумісності функцій](runtime-options-overview#options-compatibility-table) для деталей.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">